# 🎙️ Deepfake Detection Pipeline

**Steps:**
1. Securely Clone Repo
2. Install Dependencies
3. Download/Link Data
4. Run Audio Generation
5. Extract Features


## 1️⃣ Secure Setup & Clone

In [ ]:
# Mount Drive
from google.colab import drive
from getpass import getpass
import os
import shutil

drive.mount('/content/drive')

# --- CONFIGURATION ---
USERNAME = "charlesgrube-jpg"
REPO_NAME = "Data-Management-DDAA-KAN"
BRANCH = "feature/tts-vc-extraction"
# ---------------------

print("Enter your GitHub Personal Access Token (Permissions: repo):")
token = getpass()

# Secure URL construction
repo_url = f"https://{token}@github.com/{USERNAME}/{REPO_NAME}.git"

if not os.path.exists(REPO_NAME):
    !git clone {repo_url} {REPO_NAME}
    %cd {REPO_NAME}
    !git checkout {BRANCH}
    print(f"✅ Cloned and checked out {BRANCH}!")
else:
    %cd {REPO_NAME}
    print("✅ Repo already exists.")

In [ ]:
# Install Deps
!apt-get install -y ffmpeg espeak-ng
# Use %pip to ensure installation in current kernel
%pip install -r requirements_colab.txt

## 2️⃣ Data Setup

In [ ]:
# --- IMPORTANT: DATA SOURCE CONFIGURATION ---
# If your data is in a synced folder (e.g. "Computers/My Laptop/en"),
# use the file browser on the left to find it, Right Click -> Copy Path, and paste it below.
# Leave empty to try auto-detection in standard locations.
CUSTOM_SOURCE_PATH = ""  # e.g. "/content/drive/Othercomputers/My Laptop/en"
# ---------------------------------------------

TARGET_DIR = "/content/DDAA-KAN/mozilla_cv_data"

def setup_data():
    # 1. Check Custom Path First
    if CUSTOM_SOURCE_PATH and os.path.exists(CUSTOM_SOURCE_PATH):
        source = CUSTOM_SOURCE_PATH
        print(f"📂 Found custom source: {source}")
        
        # Check if it's a folder or a tar/zip
        if os.path.isdir(source):
            print("🚀 Copying data to local disk... (This ensures your Drive files remain READ-ONLY and untouched)")
            # Copying is safer and faster for processing than reading from Drive directly
            if os.path.exists(TARGET_DIR):
                 shutil.rmtree(TARGET_DIR)
            shutil.copytree(source, TARGET_DIR)
            print("✅ Data copied successfully!")
            return
        elif source.endswith(".tar.gz"):
            print("📦 Extracting custom tarball to local disk...")
            !mkdir -p {TARGET_DIR}
            !tar -xzf "{source}" -C {TARGET_DIR}
            print("✅ Extraction complete!")
            return

    # 2. Check Standard Drive Locations (Zip or Tar)
    drive_zip = "/content/drive/MyDrive/mozilla_cv_data.zip"
    drive_tar = "/content/drive/MyDrive/common_voice_sample.tar.gz"
    
    if os.path.exists(drive_zip):
        print("📦 Found Zip in Drive. Unzipping to local disk...")
        !unzip -q "{drive_zip}" -d .
        print("✅ Unzipped successfully!")
    elif os.path.exists(drive_tar):
        print("📦 Found Tar in Drive. Extracting to local disk...")
        !mkdir -p mozilla_cv_data
        !tar -xzf "{drive_tar}" -C mozilla_cv_data
        print("✅ Extracted successfully!")
    elif os.path.exists("/content/drive/MyDrive/mozilla_cv_data"):
        print("📂 Found folder in Drive. Copying to local disk for safety/speed...")
        src = "/content/drive/MyDrive/mozilla_cv_data"
        if os.path.exists("mozilla_cv_data"):
             shutil.rmtree("mozilla_cv_data")
        shutil.copytree(src, "mozilla_cv_data")
        print("✅ Copy complete!")
    else:
        print("⚠️ No data found! Please set CUSTOM_SOURCE_PATH above to your synced folder.")

setup_data()

## 3️⃣ Run Pipeline (Generate Audio)

In [ ]:
import yaml

OUTPUT_PATH = "/content/drive/MyDrive/DDAA_Pipeline_Output"

with open("config.yaml", "r") as f:
    config = yaml.safe_load(f)

# Configure for Colab
config['output']['base_dir'] = OUTPUT_PATH
config['synthesis']['tts_models'] = []
config['codec_compression']['enabled'] = True

with open("config.yaml", "w") as f:
    yaml.dump(config, f)

!python run_pipeline.py

## 4️⃣ Extract Features

In [ ]:
import os
import glob

# 1. Find the latest output folder in Drive
drive_output_pattern = "/content/drive/MyDrive/DDAA_Pipeline_Output_*"
list_of_folders = glob.glob(drive_output_pattern)

if not list_of_folders:
    print("❌ No output folder found! Did the pipeline run successfully?")
else:
    latest_folder = max(list_of_folders, key=os.path.getctime)
    print(f"✅ Found latest dataset: {latest_folder}")
    
    # 2. Define output feature folder inside the dataset folder
    output_features = os.path.join(latest_folder, "features_cqt")

    print(f"🚀 Extracting CQT features to: {output_features}")
    
    # 3. Run Extraction
    !python -m pipeline.features.extract_features \
        --type cqt \
        --input "{latest_folder}" \
        --output "{output_features}"